In [25]:
pip install fredapi python-dotenv pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from fredapi import Fred
from dotenv import load_dotenv
import pandas as pd
import os

In [28]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
fred_auth = fred = Fred(api_key=FRED_API_KEY)

In [ ]:
recession_data = fred_auth.get_series("USREC")
# most updated data is Apr 2026
recession_data.tail(5)

2025-12-01    0.0
2026-01-01    0.0
2026-02-01    0.0
2026-03-01    0.0
2026-04-01    0.0
dtype: float64

In [64]:
# list of metrics to pull from FRED: consumer price index, payroll employment, unemployment rate, industrial production, 10 year minus 3 month treasury yield
# 'realtime_start' = The Publication Day
# 'date' = The Observation Month
# 'value' = The CPI value at that time
monthly_revised = ["CPIAUCSL", "PAYEMS", "UNRATE", "INDPRO"]
daily_market = ["T10Y3M"]

economic_data = {}

for metric in monthly_revised:
    print(f"Pulling first-release data for {metric}...")
    df = fred_auth.get_series_all_releases(
        metric, 
        realtime_start="2001-01-01"
    )
    # The reason for first releases is because we are assessing the data as it was available at the time of release
    # because this often guides future economic decisions and market reactions. 
    first_release_df = df.groupby('date').head(1)
    economic_data[metric] = first_release_df

for metric in daily_market:
    print(f"Pulling market data for {metric}...")
    economic_data[metric] = fred_auth.get_series(
        metric, 
        observation_start="2001-01-01"
    )

Pulling first-release data for CPIAUCSL...
Pulling first-release data for PAYEMS...
Pulling first-release data for UNRATE...
Pulling first-release data for INDPRO...
Pulling market data for T10Y3M...


In [ ]:
economic_data["T10Y3M"].tail(5)
# we need to restructure this for the monthly cadence of the other data, so we will take the last value of each month to represent the market data for that month.

2026-05-04    0.75
2026-05-05    0.74
2026-05-06    0.67
2026-05-07    0.72
2026-05-08    0.69
dtype: float64